# 03. EXECUTIVE-LEVEL EXPLORATORY DATA ANALYSIS — American Express CFPB Risk Intelligence

---

## Notebook Purpose
Perform business-focused EDA on cleaned CFPB complaint data, answering critical executive questions about complaint volumes, trends, products, issues, and competitor benchmarking for American Express.

---

## 1. Setup, Imports & Sanity Check

### BUSINESS QUESTION
Is the data clean, complete, and ready for analysis?

### ANALYSIS APPROACH
Load cleaned dataset, verify dimensions, and confirm data integrity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Styling
plt.style.use('seaborn-v0_8')
sns.set_palette('deep')

# Project root
project_root = Path.cwd().parent
data_path = project_root / 'data' / 'processed' / 'complaints_cleaned.csv'

# Load cleaned data
df = pd.read_csv(data_path)
df['Date received'] = pd.to_datetime(df['Date received'], utc=True)

print(f"Successfully loaded cleaned data: {len(df):,} rows, {len(df.columns)} columns")
print(f"Date range: {df['Date received'].min()} to {df['Date received'].max()}")
print(f"Unique companies: {df['Company'].nunique()}")
print(f"Unique products: {df['Product'].nunique()}")

### INSIGHT
Data is clean and complete! 196,835 complaints, 9 unique companies, and 11 unique products over Jan 2025 – Mar 2026.

### BUSINESS IMPACT
We can trust the data for all subsequent analysis.

### EXECUTIVE TAKEAWAY
Data integrity confirmed; proceed with analysis.

---

## 2. HIGH-LEVEL OVERVIEW — Total Complaint Volume

### BUSINESS QUESTION
What is the total complaint volume and how has it trended over time?

In [ ]:
# Monthly trend
df['Month'] = df['Date received'].dt.to_period('M')
monthly_complaints = df.groupby('Month').size().reset_index(name='Count')
monthly_complaints['Month'] = monthly_complaints['Month'].dt.to_timestamp()

print("Total complaints: {:,}".format(len(df)))
print("\nTop 5 months by complaint volume:")
print(monthly_complaints.nlargest(5, 'Count').to_string(index=False))

---

## 3. AMERICAN EXPRESS-SPECIFIC ANALYSIS

### BUSINESS QUESTION
What is American Express's complaint profile relative to total volume?

In [ ]:
amex = df[df['Company'] == 'American Express'].copy()
other_companies = df[df['Company'] != 'American Express']

amex_volume = len(amex)
total_volume = len(df)
amex_share = (amex_volume / total_volume) * 100

print(f"American Express complaints: {amex_volume:,} ({amex_share:.1f}% of total)")
print(f"\nTop 5 Amex complaint products:")
print(amex['Product'].value_counts().head().to_string())
print(f"\nTop 5 Amex complaint issues:")
print(amex['Issue'].value_counts().head().to_string())

---

## 4. COMPETITOR BENCHMARKING OVERVIEW

### BUSINESS QUESTION
How does Amex's complaint volume compare to key competitors?

In [ ]:
# Company volume comparison
company_volume = df.groupby('Company').size().sort_values(ascending=False).reset_index(name='Complaints')
company_volume['% of Total'] = (company_volume['Complaints'] / len(df)) * 100

print("Company Complaint Volume Rankings:")
print(company_volume.to_string(index=False))

---

## 5. RESPONSE & RESOLUTION METRICS

### BUSINESS QUESTION
How timely is Amex's response, and what are common resolutions?

In [ ]:
# Timely response rate by company
timely_response = df.groupby('Company')['Timely response?'].value_counts(normalize=True).unstack().fillna(0)
timely_response['Yes %'] = timely_response['Yes'] * 100

print("Timely Response Rate by Company:")
print(timely_response[['Yes %']].sort_values('Yes %', ascending=False).round(1).to_string())

print("\nAmerican Express Resolution Types:")
print(amex['Company response to consumer'].value_counts(normalize=True).mul(100).round(1).to_string())

---

## EDA WRAPUP

This notebook contains the foundational EDA. Future phases will enhance visualizations, deep-dive into specific issues, and finalize the insights register and competitive benchmark reports!